In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from organelle_mapping.config import DataConfig
from organelle_mapping.config.run import load_subconfig

In [ ]:
# Display settings - show all rows and columns
import pandas as pd
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)  # Auto-detect width
pd.set_option('display.max_colwidth', None)  # Show full column content

In [ ]:
# Load histogram data
csv_path = "/experiments/run11/all/histograms.csv"
df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} rows")
print(f"Datasets: {df['dataset'].unique()}")
print(f"Crops per dataset:")
print(df.groupby('dataset')['crop'].nunique())

In [ ]:
data_config_path = "/experiments/run11/all/data_8nm_org+mem.yaml"
config = load_subconfig(data_config_path, DataConfig)

# Create dataframe with contrast info
contrast_info = []
for ds_name, ds_info in config.datasets.items():
    minc, maxc = ds_info.em.contrast
    
    # Calculate what minc and maxc map to after CORRECT normalization
    # Formula: ((intensity - minc) / (maxc - minc)) * 2 - 1
    # At minc: ((minc - minc) / (maxc - minc)) * 2 - 1 = 0 * 2 - 1 = -1
    # At maxc: ((maxc - minc) / (maxc - minc)) * 2 - 1 = 1 * 2 - 1 = 1
    norm_contrast_min = -1.0
    norm_contrast_max = 1.0
    
    contrast_info.append({
        'dataset': ds_name,
        'contrast_min': minc,
        'contrast_max': maxc,
        'contrast_range': maxc - minc,
        'norm_contrast_min': norm_contrast_min,
        'norm_contrast_max': norm_contrast_max,
        'norm_contrast_range': norm_contrast_max - norm_contrast_min
    })

contrast_df = pd.DataFrame(contrast_info)
print("\nDataset contrast ranges:")
contrast_df

In [ ]:
# Apply CORRECTED normalization pipeline to add normalized columns
def apply_normalization(intensity, dataset_name, contrast_df):
    """Apply the CORRECT normalization pipeline:
    Map [minc, maxc] to [-1, 1]
    
    Formula: ((intensity - minc) / (maxc - minc)) * 2 - 1
    """
    # Look up contrast values from dataframe
    ds_row = contrast_df[contrast_df['dataset'] == dataset_name].iloc[0]
    minc = ds_row['contrast_min']
    maxc = ds_row['contrast_max']
    
    # Map [minc, maxc] to [-1, 1]
    normalized = ((intensity - minc) / (maxc - minc)) * 2 - 1
    
    return normalized

# Add normalized intensity as a new column
df['normalized_intensity'] = df.apply(
    lambda row: apply_normalization(row['intensity'], row['dataset'], contrast_df),
    axis=1
)

print(f"Added normalized_intensity column")
print(f"Normalized intensity range: [{df['normalized_intensity'].min():.3f}, {df['normalized_intensity'].max():.3f}]")

In [ ]:
# Summary statistics for normalized data
def compute_norm_stats(group):
    intensities = np.repeat(group['normalized_intensity'].values, group['count'].values)
    return pd.Series({
        'norm_mean': intensities.mean(),
        'norm_std': intensities.std(),
        'norm_min': intensities.min(),
        'norm_max': intensities.max(),
        'norm_p01': np.percentile(intensities, 1),
        'norm_p99': np.percentile(intensities, 99),
        'norm_median': np.median(intensities)
    })

norm_stats = df.groupby(['dataset', 'crop']).apply(compute_norm_stats, include_groups=False).reset_index()
norm_stats['norm_range'] = norm_stats['norm_max'] - norm_stats['norm_min']

# Add normalized contrast values from config
norm_stats = norm_stats.merge(contrast_df[['dataset', 'norm_contrast_min', 'norm_contrast_max']], on='dataset', how='left')

# Find crops with narrow normalized ranges
narrow_threshold = 0.1  # normalized intensity range threshold
narrow_norm_crops = norm_stats[norm_stats['norm_range'] < narrow_threshold].sort_values('norm_range')
print(f"\nFound {len(narrow_norm_crops)} crops with normalized range < {narrow_threshold}:")
print("\nNarrow crops (sorted by range):")
narrow_norm_crops[['dataset', 'crop', 'norm_min', 'norm_max', 'norm_range', 'norm_mean', 'norm_std', 'norm_contrast_min', 'norm_contrast_max']]

In [ ]:
norm_stats

In [ ]:
# Summary statistics per crop (raw intensities)
def compute_stats(group):
    intensities = np.repeat(group['intensity'].values, group['count'].values)
    return pd.Series({
        'mean': intensities.mean(),
        'std': intensities.std(),
        'min': intensities.min(),
        'max': intensities.max(),
        'p01': np.percentile(intensities, 1),
        'p99': np.percentile(intensities, 99),
        'median': np.median(intensities)
    })

stats = df.groupby(['dataset', 'crop']).apply(compute_stats, include_groups=False).reset_index()
stats['range'] = stats['max'] - stats['min']

# Add contrast values from config
stats = stats.merge(contrast_df[['dataset', 'contrast_min', 'contrast_max']], on='dataset', how='left')

stats

In [ ]:
# Interactive crop browser
import ipywidgets as widgets
from IPython.display import display, clear_output

# Get all dataset/crop combinations
all_crops = []
for dataset in df['dataset'].unique():
    crops = df[df['dataset'] == dataset]['crop'].unique()
    for crop in crops:
        all_crops.append((dataset, crop))

# Create stats lookup for quick access
stats_lookup = stats.set_index(['dataset', 'crop'])
norm_stats_lookup = norm_stats.set_index(['dataset', 'crop'])

# Current index
current_idx = [0]  # Using list to allow modification in nested function

def show_crop(idx):
    """Display histograms and stats for a given crop index."""
    dataset_name, crop_name = all_crops[idx]
    
    # Get data for this crop
    crop_data = df[(df['dataset'] == dataset_name) & (df['crop'] == crop_name)].sort_values('intensity')
    crop_data_norm = crop_data.sort_values('normalized_intensity')
    
    # Calculate bar widths
    width_raw = 1.0  # Raw intensities are integers
    intensities_norm = crop_data_norm['normalized_intensity'].unique()
    width_norm = intensities_norm[1] - intensities_norm[0] if len(intensities_norm) > 1 else 0.01
    
    # Get stats
    crop_stats = stats_lookup.loc[(dataset_name, crop_name)]
    crop_norm_stats = norm_stats_lookup.loc[(dataset_name, crop_name)]
    
    # Create figure with 2 subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    
    # Raw histogram
    ax1.bar(crop_data['intensity'], crop_data['count'], width=width_raw, edgecolor='none', align='edge')
    ax1.set_xlabel('Raw Intensity')
    ax1.set_ylabel('Count')
    ax1.set_title(f'{dataset_name} - {crop_name} (Raw)')
    ax1.set_xlim(0, 255)
    ax1.axvline(x=0, color="red", linestyle="--", alpha=0.3, label="Max range")
    ax1.axvline(x=255, color="red", linestyle="--", alpha=0.3)
    ax1.axvline(x=crop_stats['contrast_min'], color='green', linestyle='--', alpha=0.5, label='Contrast range')
    ax1.axvline(x=crop_stats['contrast_max'], color='green', linestyle='--', alpha=0.5)
    ax1.legend()
    
    # Normalized histogram
    ax2.bar(crop_data_norm['normalized_intensity'], crop_data_norm['count'], width=width_norm, edgecolor='none', align='edge')
    ax2.set_xlabel('Normalized Intensity')
    ax2.set_ylabel('Count')
    ax2.set_title(f'{dataset_name} - {crop_name} (Normalized)')
    ax2.set_xlim(-1, 1)
    ax2.axvline(x=-1, color='red', linestyle='--', alpha=0.3, label='Max range')
    ax2.axvline(x=1, color='red', linestyle='--', alpha=0.3)
    ax2.axvline(x=crop_norm_stats['norm_contrast_min'], color='green', linestyle='--', alpha=0.5, label='Contrast range')
    ax2.axvline(x=crop_norm_stats['norm_contrast_max'], color='green', linestyle='--', alpha=0.5)
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Display stats
    print("=" * 80)
    print(f"RAW STATISTICS - {dataset_name} / {crop_name}")
    print("=" * 80)
    print(f"  Range:       [{crop_stats['min']:.1f}, {crop_stats['max']:.1f}]  (span: {crop_stats['range']:.1f})")
    print(f"  Mean:        {crop_stats['mean']:.2f}")
    print(f"  Std:         {crop_stats['std']:.2f}")
    print(f"  Median:      {crop_stats['median']:.1f}")
    print(f"  Percentiles: p01={crop_stats['p01']:.1f}, p99={crop_stats['p99']:.1f}")
    print(f"  Contrast:    [{crop_stats['contrast_min']}, {crop_stats['contrast_max']}]")
    print()
    print("=" * 80)
    print(f"NORMALIZED STATISTICS - {dataset_name} / {crop_name}")
    print("=" * 80)
    print(f"  Range:       [{crop_norm_stats['norm_min']:.4f}, {crop_norm_stats['norm_max']:.4f}]  (span: {crop_norm_stats['norm_range']:.4f})")
    print(f"  Mean:        {crop_norm_stats['norm_mean']:.4f}")
    print(f"  Std:         {crop_norm_stats['norm_std']:.6f}")
    print(f"  Median:      {crop_norm_stats['norm_median']:.4f}")
    print(f"  Percentiles: p01={crop_norm_stats['norm_p01']:.4f}, p99={crop_norm_stats['norm_p99']:.4f}")
    print(f"  Contrast:    [{crop_norm_stats['norm_contrast_min']:.4f}, {crop_norm_stats['norm_contrast_max']:.4f}]")
    print("=" * 80)

def on_prev_clicked(b):
    """Handle previous button click."""
    if current_idx[0] > 0:
        current_idx[0] -= 1
        output.clear_output(wait=True)
        with output:
            label.value = f"Crop {current_idx[0] + 1} of {len(all_crops)}"
            show_crop(current_idx[0])

def on_next_clicked(b):
    """Handle next button click."""
    if current_idx[0] < len(all_crops) - 1:
        current_idx[0] += 1
        output.clear_output(wait=True)
        with output:
            label.value = f"Crop {current_idx[0] + 1} of {len(all_crops)}"
            show_crop(current_idx[0])

def on_dropdown_change(change):
    """Handle dropdown selection change."""
    current_idx[0] = change['new']
    output.clear_output(wait=True)
    with output:
        label.value = f"Crop {current_idx[0] + 1} of {len(all_crops)}"
        show_crop(current_idx[0])

# Create widgets
prev_button = widgets.Button(description="← Previous")
next_button = widgets.Button(description="Next →")
label = widgets.Label(value=f"Crop 1 of {len(all_crops)}")

# Create dropdown with dataset/crop names
dropdown_options = [(f"{ds} / {crop}", idx) for idx, (ds, crop) in enumerate(all_crops)]
dropdown = widgets.Dropdown(
    options=dropdown_options,
    value=0,
    description='Select:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# Connect event handlers
prev_button.on_click(on_prev_clicked)
next_button.on_click(on_next_clicked)
dropdown.observe(on_dropdown_change, names='value')

# Create output widget
output = widgets.Output()

# Display controls
controls = widgets.HBox([prev_button, label, next_button, dropdown])
display(controls)
display(output)

# Show initial crop
with output:
    show_crop(0)